# Intelligent Neurons Network

In [13]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import random
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os
import time

## 1. Le Neurone "Intelligent" (Micro-Architecture)
L’idée est de donner une forme d’intelligence aux neurones :

Chaque neurone doit se spécialiser dans une tâche dans un certain contexte. Sauf que selon le contexte, il a besoin d’information différentes. On munit donc chaque neurone de la structure suivante

### Signal Receiver (Input Interface)
Il agit comme un filtre. Le vecteur Query détermine ce que le neurone cherche à écouter. Le Feed Forward (FF) ici sert probablement de pré-traitement ou d'encodage de l'entrée brute.

### Cœur (Memory/Processing)
Le LSTM est le moteur. Il maintient un état interne ($h_t, c_t$) qui évolue dans le temps. Cela permet au neurone de retenir un contexte sur plusieurs pas de temps.

### Signal Sender (Output Interface)
Il agit comme un diffuseur sélectif. Le vecteur Key définit le "contenu" ou le "type" d'information que le neurone propose. Le FF de sortie formate l'état caché du LSTM vers un signal transmissible.

In [2]:
class IntelligentNeuron(nn.Module):
    def __init__(self, input_dim, hidden_dim, key_query_dim, output_dim=None):
        """
        Initialise un 'Intelligent Neuron'.

        Args:
            input_dim (int): Dimension du signal d'entrée brut (avant le Feed Forward du receiver).
                             Si c'est un neurone de transmission, cela correspond souvent à la dimension de sortie des autres neurones.
            hidden_dim (int): Dimension de la mémoire interne (état caché du LSTM).
            key_query_dim (int): Dimension des vecteurs Key et Query pour le mécanisme d'attention.
            output_dim (int, optional): Dimension du signal de sortie. Si None, par défaut à hidden_dim.
        """
        super(IntelligentNeuron, self).__init__()

        self.hidden_dim = hidden_dim
        self.output_dim = output_dim if output_dim is not None else hidden_dim

        # --- Signal Receiver ---
        # Vecteur Query : Ce que le neurone "veut" écouter.
        # On l'initialise comme un paramètre apprenable.
        self.query = nn.Parameter(torch.randn(key_query_dim))

        # Receiver Feed Forward : Prépare l'input pour le LSTM.
        self.receiver_ff = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU() # Activation non-linéaire
        )

        # --- Core (Mémoire) ---
        # LSTM Cell : Gère la mémoire à long terme et l'état courant.
        # input_size est hidden_dim car le receiver_ff a déjà transformé l'input.
        self.lstm_cell = nn.LSTMCell(input_size=hidden_dim, hidden_size=hidden_dim)

        # --- Signal Sender ---
        # Vecteur Key : Ce que le neurone "annonce" contenir comme information.
        self.key = nn.Parameter(torch.randn(key_query_dim))

        # Sender Feed Forward : Prépare la sortie du LSTM pour être envoyée.
        self.sender_ff = nn.Sequential(
            nn.Linear(hidden_dim, self.output_dim),
            nn.Identity() # On peut ajouter une activation ici si nécessaire (ex: Tanh, Sigmoid)
        )

        # État interne (h_t, c_t) initialisé à None (sera initialisé au premier forward)
        self.h_t = None
        self.c_t = None

    def init_state(self, batch_size=1):
        """Réinitialise l'état interne (mémoire) du neurone."""
        device = self.query.device
        self.h_t = torch.zeros(batch_size, self.hidden_dim).to(device)
        self.c_t = torch.zeros(batch_size, self.hidden_dim).to(device)

    def forward(self, input_signal):
        """
        Effectue un pas de temps pour le neurone.

        Args:
            input_signal (torch.Tensor): Le signal d'entrée agrégé (batch_size, input_dim).
                                         Note: L'agrégation (filtrage par attention) est généralement
                                         gérée par le réseau avant d'appeler ce forward,
                                         car elle dépend des Keys des autres neurones.

        Returns:
            output_signal (torch.Tensor): Le signal émis par le neurone (batch_size, output_dim).
            query (torch.Tensor): Le vecteur query actuel du neurone (pour le prochain pas de filtrage).
            key (torch.Tensor): Le vecteur key actuel du neurone.
        """
        batch_size = input_signal.size(0)

        # Initialisation de l'état si nécessaire
        if self.h_t is None or self.h_t.size(0) != batch_size:
            self.init_state(batch_size)

        # 1. Receiver : Traitement de l'input
        processed_input = self.receiver_ff(input_signal)

        # 2. Core : Mise à jour de la mémoire LSTM
        self.h_t, self.c_t = self.lstm_cell(processed_input, (self.h_t, self.c_t))

        # 3. Sender : Génération de l'output
        output_signal = self.sender_ff(self.h_t)

        # Note: Dans cette architecture simple, Key et Query sont des vecteurs statiques (paramètres).
        # On pourrait imaginer qu'ils soient dynamiques (générés par le LSTM),
        # mais pour l'instant on suit la définition de base.
        # On les retourne pour faciliter le calcul d'attention au niveau du réseau.

        # Expansion des dimensions pour correspondre au batch_size si nécessaire
        # (batch_size, key_query_dim)
        current_key = self.key.unsqueeze(0).expand(batch_size, -1)
        current_query = self.query.unsqueeze(0).expand(batch_size, -1)

        return output_signal, current_query, current_key

## 2. Le Réseau (Macro-Architecture)
###Topologie
C'est un graphe connexe. Contrairement aux couches denses classiques, la communication dépend de la compatibilité Key-Query.

###Mécanisme d'Attention Distribué
L'interaction $Query_{receveur} \cdot Key_{émetteur}$ définit le poids de la connexion (l'attention). Si le produit scalaire est élevé, le neurone A "écoute" fortement le neurone B. Cela permet au réseau d'apprendre sa propre topologie dynamique (plasticité).

###Types de Neurones
- Input : Encodeurs sensoriels.
- Transmission : "Hidden layers" dynamiques.
- Action : Décodeurs/Actuateurs.

In [3]:
class IntelligentNetwork(nn.Module):
    def __init__(self,
                 input_size,
                 output_size,
                 num_input_neurons,
                 num_transmission_neurons,
                 num_action_neurons,
                 neuron_hidden_dim=32,
                 key_query_dim=16,
                 signal_dim=16):
        """
        Réseau de neurones intelligents.

        Args:
            input_size (int): Dimension globale de l'entrée du système.
            output_size (int): Dimension globale de la sortie du système.
            num_input_neurons (int): Nombre de neurones dédiés à la réception de l'entrée.
            num_transmission_neurons (int): Nombre de neurones cachés de traitement.
            num_action_neurons (int): Nombre de neurones produisant la sortie finale.
            neuron_hidden_dim (int): Taille de la mémoire interne (LSTM) de chaque neurone.
            key_query_dim (int): Taille des vecteurs d'attention.
            signal_dim (int): Taille du vecteur échangé entre les neurones.
        """
        super(IntelligentNetwork, self).__init__()

        self.num_input_neurons = num_input_neurons
        self.num_transmission_neurons = num_transmission_neurons
        self.num_action_neurons = num_action_neurons

        self.signal_dim = signal_dim
        self.key_query_dim = key_query_dim

        # --- Création des Neurones ---
        self.neurons = nn.ModuleList()

        # 1. Input Neurons
        # Ils reçoivent l'input externe directement.
        for _ in range(num_input_neurons):
            self.neurons.append(IntelligentNeuron(input_dim=input_size,
                                                  hidden_dim=neuron_hidden_dim,
                                                  key_query_dim=key_query_dim,
                                                  output_dim=signal_dim))

        # 2. Transmission Neurons
        # Ils reçoivent des signaux d'autres neurones (taille signal_dim).
        for _ in range(num_transmission_neurons):
            self.neurons.append(IntelligentNeuron(input_dim=signal_dim,
                                                  hidden_dim=neuron_hidden_dim,
                                                  key_query_dim=key_query_dim,
                                                  output_dim=signal_dim))

        # 3. Action Neurons
        # Ils reçoivent des signaux internes et produisent un signal interne + contribuent à la sortie.
        for _ in range(num_action_neurons):
            self.neurons.append(IntelligentNeuron(input_dim=signal_dim,
                                                  hidden_dim=neuron_hidden_dim,
                                                  key_query_dim=key_query_dim,
                                                  output_dim=signal_dim))

        # Projection finale pour les Action Neurons vers la sortie système
        self.action_projection = nn.Linear(num_action_neurons * signal_dim, output_size)

        # Stockage des outputs précédents pour la récurrence (mémoire tampon)
        self.previous_outputs = None

    def reset_memory(self, batch_size):
        """Réinitialise la mémoire de tous les neurones et le buffer d'outputs."""
        for neuron in self.neurons:
            neuron.init_state(batch_size)

        device = self.neurons[0].query.device
        total_neurons = len(self.neurons)
        # Initialisation à 0 des signaux précédents
        self.previous_outputs = torch.zeros(batch_size, total_neurons, self.signal_dim).to(device)

    def forward(self, x):
        """
        Passe avant (Forward pass) sur un pas de temps ou une séquence.

        Args:
            x: Input externe. (Batch, Input_Size) ou (Batch, Time, Input_Size)
        """
        if x.dim() == 2: # (Batch, Input_Size) -> Un seul pas de temps
            return self.step(x)
        elif x.dim() == 3: # (Batch, Time, Input_Size) -> Séquence
            outputs = []
            # On reset la mémoire au début d'une nouvelle séquence
            self.reset_memory(x.size(0))

            for t in range(x.size(1)):
                input_t = x[:, t, :]
                out_t = self.step(input_t)
                outputs.append(out_t.unsqueeze(1)) # Ajouter dimension temps

            return torch.cat(outputs, dim=1) # (Batch, Time, Output_Size)
        else:
            raise ValueError("Input doit être 2D (step) ou 3D (sequence)")

    def step(self, x_input):
        """Exécute un pas de temps t."""
        batch_size = x_input.size(0)

        # Si premier pas, initialisation
        if self.previous_outputs is None or self.previous_outputs.size(0) != batch_size:
            self.reset_memory(batch_size)

        # --- 1. Récupération des Keys et Queries ---
        all_queries = []
        all_keys = []

        for neuron in self.neurons:
            q = neuron.query.unsqueeze(0).expand(batch_size, -1)
            k = neuron.key.unsqueeze(0).expand(batch_size, -1)
            all_queries.append(q)
            all_keys.append(k)

        Q_matrix = torch.stack(all_queries, dim=1) # (Batch, N, KQ_Dim)
        K_matrix = torch.stack(all_keys, dim=1)    # (Batch, N, KQ_Dim)

        # --- 2. Calcul de l'Attention ---
        # Score(i, j) : i écoute j
        attention_scores = torch.bmm(Q_matrix, K_matrix.transpose(1, 2))
        attention_scores = attention_scores / (self.key_query_dim ** 0.5)
        attention_weights = F.softmax(attention_scores, dim=2)

        # --- 3. Agrégation des signaux ---
        context_inputs = torch.bmm(attention_weights, self.previous_outputs)

        # --- 4. Forward des Neurones ---
        new_outputs = []
        idx_input_end = self.num_input_neurons

        for i, neuron in enumerate(self.neurons):
            if i < idx_input_end:
                # Input Neurons : Input externe
                inp = x_input
            else:
                # Autres : Contexte agrégé
                inp = context_inputs[:, i, :]

            out_signal, _, _ = neuron(inp)
            new_outputs.append(out_signal)

        self.previous_outputs = torch.stack(new_outputs, dim=1)

        # --- 5. Sortie Système ---
        # On récupère les outputs des derniers (Action Neurons)
        start_action = self.num_input_neurons + self.num_transmission_neurons
        action_outputs = self.previous_outputs[:, start_action:, :]

        action_outputs_flat = action_outputs.reshape(batch_size, -1)
        system_output = self.action_projection(action_outputs_flat)

        return system_output

## Génération de texte Shakespear-like

En reprenant l'exemple de la vidéo d'Andrej Karpathy qui présente l'architecture d'un transformer, voyons si notre architecture est capable de réaliser cette génération.

Ce n'est pas vraiment sur ce genre d'exemple que l'INN est le plus pertinent car l'intérêt est de pouvoir mélanger plusieurs types d'input, traités complètement différemment, et de réaliser plusieurs actions totalement différentes également.

In [4]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2025-11-22 20:07:41--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.03s   

2025-11-22 20:07:42 (37.9 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [5]:
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [ ]:
def plot_attention_matrix(model, filename="attention_matrix.png", title="Matrice d'Attention des Neurones"):
    """
    Récupère les Queries et Keys actuelles du modèle et affiche la matrice d'attention.
    Axe Y : Receveurs (Qui écoute ?)
    Axe X : Emetteurs (Qui parle ?)
    """
    model.eval()

    device = next(model.parameters()).device

    with torch.no_grad():
        all_queries = []
        all_keys = []

        for neuron in model.neurons:
            q = neuron.query.unsqueeze(0)
            k = neuron.key.unsqueeze(0)
            all_queries.append(q)
            all_keys.append(k)

        Q_matrix = torch.stack(all_queries, dim=1).to(device)
        K_matrix = torch.stack(all_keys, dim=1).to(device)

        scores = torch.bmm(Q_matrix, K_matrix.transpose(1, 2))
        scores = scores / (model.key_query_dim ** 0.5)
        attention_weights = torch.nn.functional.softmax(scores, dim=2)

        attention_matrix = attention_weights[0].cpu().numpy()

    plt.figure(figsize=(14, 12))

    labels = []
    # Input
    for i in range(model.num_input_neurons):
        labels.append(f"I{i}")
    # Trans
    for i in range(model.num_transmission_neurons):
        labels.append(f"T{i}")
    # Action
    for i in range(model.num_action_neurons):
        labels.append(f"A{i}")

    sns.heatmap(attention_matrix, xticklabels=labels, yticklabels=labels, cmap="viridis", vmin=0, vmax=1)

    plt.title(title)
    plt.xlabel("Émetteur (Source)")
    plt.ylabel("Receveur (Destination)")
    plt.tight_layout()

    # Créer le dossier plots s'il n'existe pas
    if os.path.dirname(filename) and not os.path.exists(os.path.dirname(filename)):
        os.makedirs(os.path.dirname(filename))

    plt.savefig(filename)
    plt.close()
    print(f"Matrice d'attention sauvegardée dans '{filename}'")



In [ ]:
# --- Configuration ---
BATCH_SIZE = 32
SEQ_LEN = 64
LEARNING_RATE = 0.001  # Réduit pour meilleure convergence
EPOCHS = 20           # Augmenté
HIDDEN_DIM = 256      # Augmenté
KEY_QUERY_DIM = 256   # Augmenté
SIGNAL_DIM = 256      # Augmenté pour améliorer la communication
NUM_INPUT = 5
NUM_TRANS = 40        # Fortement augmenté pour donner de la capacité de calcul
NUM_ACTION = 5
PRINT_EVERY = 100

def load_data(filepath):
    # On suppose que le fichier existe et est accessible
    #with open(filepath, 'r', encoding='utf-8') as f:
    #    text = f.read()
    chars = sorted(list(set(text)))
    vocab_size = len(chars)
    char2idx = {ch: i for i, ch in enumerate(chars)}
    idx2char = {i: ch for i, ch in enumerate(chars)}
    print(f"Texte chargé : {len(text)} caractères")
    print(f"Vocabulaire : {vocab_size} caractères uniques")
    return text, chars, char2idx, idx2char, vocab_size

def get_batch(text, char2idx, batch_size, seq_len, vocab_size):
    input_seqs = []
    target_seqs = []

    for _ in range(batch_size):
        start_idx = random.randint(0, len(text) - seq_len - 1)
        chunk = text[start_idx : start_idx + seq_len + 1]

        # Input : One-hot encoding
        input_indices = [char2idx[c] for c in chunk[:-1]]
        target_indices = [char2idx[c] for c in chunk[1:]]

        # Conversion en tenseurs one-hot pour l'input
        input_tensor = torch.zeros(seq_len, vocab_size)
        for t, idx in enumerate(input_indices):
            input_tensor[t][idx] = 1.0

        input_seqs.append(input_tensor)
        target_seqs.append(torch.tensor(target_indices, dtype=torch.long))

    # Stack -> (Batch, Time, Input_Dim)
    inputs = torch.stack(input_seqs)
    # Stack -> (Batch, Time)
    targets = torch.stack(target_seqs)

    return inputs, targets

def generate_text(model, start_str, char2idx, idx2char, vocab_size, length=100, temperature=0.8):
    # Température baissée à 0.8 pour réduire le bruit
    model.eval()
    device = next(model.parameters()).device

    input_seq = torch.zeros(1, 1, vocab_size).to(device)

    # On chauffe la mémoire avec le start_str
    model.reset_memory(batch_size=1)

    current_char = start_str[0]
    input_seq[0, 0, char2idx[current_char]] = 1.0

    generated_str = start_str

    with torch.no_grad():
        # Pre-fill memory
        seed_tensor = torch.zeros(1, len(start_str), vocab_size).to(device)
        for t, char in enumerate(start_str):
            seed_tensor[0, t, char2idx[char]] = 1.0

        outputs = model(seed_tensor)
        last_output = outputs[:, -1, :]

        probs = torch.softmax(last_output / temperature, dim=1)
        next_idx = torch.multinomial(probs, 1).item()
        next_char = idx2char[next_idx]
        generated_str += next_char

        # Génération boucle
        current_input = torch.zeros(1, 1, vocab_size).to(device)
        current_input[0, 0, next_idx] = 1.0

        for _ in range(length - 1):
            output = model(current_input.squeeze(1))

            probs = torch.softmax(output / temperature, dim=1)
            next_idx = torch.multinomial(probs, 1).item()
            next_char = idx2char[next_idx]
            generated_str += next_char

            # Prepare next input
            current_input.zero_()
            current_input[0, 0, next_idx] = 1.0

    model.train()
    return generated_str

def train():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Utilisation du device : {device}")

    text, chars, char2idx, idx2char, vocab_size = load_data('test/input.txt')

    model = IntelligentNetwork(
        input_size=vocab_size,
        output_size=vocab_size,
        num_input_neurons=NUM_INPUT,
        num_transmission_neurons=NUM_TRANS,
        num_action_neurons=NUM_ACTION,
        neuron_hidden_dim=HIDDEN_DIM,
        key_query_dim=KEY_QUERY_DIM,
        signal_dim=SIGNAL_DIM
    ).to(device)

    print(f"Modèle créé avec {NUM_INPUT + NUM_TRANS + NUM_ACTION} neurones.")

    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    criterion = nn.CrossEntropyLoss()

    print("Début de l'entraînement...")

    # Création dossier plots
    if not os.path.exists("plots"):
        os.makedirs("plots")

    # On utilise plus de batches pour que l'entraînement soit plus significatif
    # (dans Colab vous pourrez ajuster si trop long, mais il faut de la donnée)
    batches_per_epoch = 200

    for epoch in range(EPOCHS):
        total_loss = 0
        for i in range(batches_per_epoch):
            inputs, targets = get_batch(text, char2idx, BATCH_SIZE, SEQ_LEN, vocab_size)
            inputs, targets = inputs.to(device), targets.to(device)

            optimizer.zero_grad()

            outputs = model(inputs)
            loss = criterion(outputs.reshape(-1, vocab_size), targets.reshape(-1))

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()

            total_loss += loss.item()

            if i % PRINT_EVERY == 0:
                print(f"Epoch {epoch+1}, Batch {i}/{batches_per_epoch}, Loss: {loss.item():.4f}")

        avg_loss = total_loss / batches_per_epoch
        print(f"=== Fin Epoch {epoch+1}, Moyenne Loss: {avg_loss:.4f} ===")

        # Visualisation
        if (epoch + 1) % 2 == 0 or epoch == 0: # Un peu plus fréquent pour voir l'évolution au début
             try:
                plot_attention_matrix(model, filename=f"plots/attention_epoch_{epoch+1}.png",
                                      title=f"Attention Matrix - Epoch {epoch+1}")
             except Exception as e:
                 print(f"Erreur visualisation: {e}")

        print("Génération de texte :")
        print(generate_text(model, "The ", char2idx, idx2char, vocab_size))
        print("===============================================================")

train()


Utilisation du device : cuda
Texte chargé : 1115394 caractères
Vocabulaire : 65 caractères uniques
Modèle créé avec 50 neurones.
Début de l'entraînement...
Epoch 1, Batch 0/200, Loss: 4.1719
Epoch 1, Batch 100/200, Loss: 3.1682
=== Fin Epoch 1, Moyenne Loss: 3.1790 ===
Matrice d'attention sauvegardée dans 'plots/attention_epoch_1.png'
Génération de texte :
The bhde  euphoesw weyn
eow srpeo yo eius ih d iyttssg hhetmss ,oer snrnogn,ftnry vn eeuil somwetouo sim
Epoch 2, Batch 0/200, Loss: 2.9181
Epoch 2, Batch 100/200, Loss: 2.8003
=== Fin Epoch 2, Moyenne Loss: 2.7924 ===
Matrice d'attention sauvegardée dans 'plots/attention_epoch_2.png'
Génération de texte :
The  fgcmlt oo  eees uad bercm snacd:hInfOllyn,hgsynma liag,tw
ee  oa  ravn 
io  eee aire.sYrs Eebhalrs 
Epoch 3, Batch 0/200, Loss: 2.7020
Epoch 3, Batch 100/200, Loss: 2.5625
=== Fin Epoch 3, Moyenne Loss: 2.6147 ===
Génération de texte :
The  rosetaithn sreibhev nocmsn
nnfsrte  oel  hetddphorsiBg
bh  av teessint phrnslrsd aoih w

KeyboardInterrupt: 

## Ajoutons maintenant le dynamisme pour que notre INN soit complet

In [15]:
class VectorizedIntelligentNetwork(nn.Module):
    def __init__(self,
                 input_size,
                 output_size,
                 num_input_neurons,
                 num_transmission_neurons, # Statiques
                 num_dynamic_neurons,      # NOUVEAU : Dynamiques
                 num_action_neurons,
                 neuron_hidden_dim=256,
                 key_query_dim=256,
                 signal_dim=256):
        super().__init__()

        self.num_input = num_input_neurons
        self.num_trans_static = num_transmission_neurons
        self.num_trans_dynamic = num_dynamic_neurons
        self.num_action = num_action_neurons

        # Ordre en mémoire : [Input, Trans_Static, Action, Trans_Dynamic]
        # 1. Les Statiques (Input + Trans_Static + Action)
        # 2. Les Dynamiques (Trans_Dynamic)

        self.num_static_total = num_input_neurons + num_transmission_neurons + num_action_neurons
        self.total_neurons = self.num_static_total + num_dynamic_neurons

        self.hidden_dim = neuron_hidden_dim
        self.key_query_dim = key_query_dim
        self.signal_dim = signal_dim

        # --- POIDS COMMUNS ---

        # 1. RECEIVER
        self.w_rec_input = nn.Parameter(torch.Tensor(num_input_neurons, input_size, neuron_hidden_dim))
        self.b_rec_input = nn.Parameter(torch.Tensor(num_input_neurons, neuron_hidden_dim))

        num_others = self.total_neurons - num_input_neurons
        self.w_rec_other = nn.Parameter(torch.Tensor(num_others, signal_dim, neuron_hidden_dim))
        self.b_rec_other = nn.Parameter(torch.Tensor(num_others, neuron_hidden_dim))

        # 2. LSTM
        self.w_lstm_ih = nn.Parameter(torch.Tensor(self.total_neurons, neuron_hidden_dim, 4 * neuron_hidden_dim))
        self.b_lstm_ih = nn.Parameter(torch.Tensor(self.total_neurons, 4 * neuron_hidden_dim))
        self.w_lstm_hh = nn.Parameter(torch.Tensor(self.total_neurons, neuron_hidden_dim, 4 * neuron_hidden_dim))
        self.b_lstm_hh = nn.Parameter(torch.Tensor(self.total_neurons, 4 * neuron_hidden_dim))

        # 3. ATTENTION HYBRIDE
        # Partie Statique (Input + Trans_Static + Action)
        self.q_static = nn.Parameter(torch.Tensor(self.num_static_total, key_query_dim))
        self.k_static = nn.Parameter(torch.Tensor(self.num_static_total, key_query_dim))

        # Partie Dynamique (Trans_Dynamic)
        if num_dynamic_neurons > 0:
            self.w_query_dyn = nn.Parameter(torch.Tensor(num_dynamic_neurons, neuron_hidden_dim, key_query_dim))
            self.b_query_dyn = nn.Parameter(torch.Tensor(num_dynamic_neurons, key_query_dim))
            self.w_key_dyn = nn.Parameter(torch.Tensor(num_dynamic_neurons, neuron_hidden_dim, key_query_dim))
            self.b_key_dyn = nn.Parameter(torch.Tensor(num_dynamic_neurons, key_query_dim))

        # 4. SENDER
        self.w_sender = nn.Parameter(torch.Tensor(self.total_neurons, neuron_hidden_dim, signal_dim))
        self.b_sender = nn.Parameter(torch.Tensor(self.total_neurons, signal_dim))

        # 5. PROJECTION
        self.action_projection = nn.Linear(num_action_neurons * signal_dim, output_size)

        self._init_weights()

        self.h_state = None
        self.c_state = None
        self.prev_outputs = None
        self.last_Q = None
        self.last_K = None

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)
            else:
                nn.init.zeros_(p)

    def reset_memory(self, batch_size):
        device = self.w_rec_input.device
        self.h_state = torch.zeros(batch_size, self.total_neurons, self.hidden_dim, device=device)
        self.c_state = torch.zeros(batch_size, self.total_neurons, self.hidden_dim, device=device)
        self.prev_outputs = torch.zeros(batch_size, self.total_neurons, self.signal_dim, device=device)

    def forward(self, x):
        if x.dim() == 2: return self.step(x)
        batch_size, seq_len, _ = x.size()
        self.reset_memory(batch_size)
        outputs = []
        for t in range(seq_len):
            outputs.append(self.step(x[:, t, :]))
        return torch.stack(outputs, dim=1)

    def step(self, x_input):
        batch_size = x_input.size(0)
        if self.h_state is None: self.reset_memory(batch_size)

        # --- 1. ATTENTION HYBRIDE ---
        Q_stat = self.q_static.unsqueeze(0).expand(batch_size, -1, -1)
        K_stat = self.k_static.unsqueeze(0).expand(batch_size, -1, -1)

        if self.num_trans_dynamic > 0:
            h_dyn = self.h_state[:, self.num_static_total:, :]
            Q_dyn = torch.einsum('bnh,nhk->bnk', h_dyn, self.w_query_dyn) + self.b_query_dyn
            K_dyn = torch.einsum('bnh,nhk->bnk', h_dyn, self.w_key_dyn) + self.b_key_dyn

            Q = torch.cat([Q_stat, Q_dyn], dim=1)
            K = torch.cat([K_stat, K_dyn], dim=1)
        else:
            Q, K = Q_stat, K_stat

        self.last_Q = Q
        self.last_K = K

        scores = torch.bmm(Q, K.transpose(1, 2)) / (self.key_query_dim ** 0.5)
        weights = F.softmax(scores, dim=2)
        context_input = torch.bmm(weights, self.prev_outputs)

        # --- 2. RECEIVER ---
        x_in_expanded = x_input.unsqueeze(1).expand(-1, self.num_input, -1)
        rec_out_input = torch.einsum('bni,nih->bnh', x_in_expanded, self.w_rec_input) + self.b_rec_input

        context_others = context_input[:, self.num_input:, :]
        rec_out_others = torch.einsum('bni,nih->bnh', context_others, self.w_rec_other) + self.b_rec_other

        lstm_input = torch.cat([rec_out_input, rec_out_others], dim=1)
        lstm_input = F.relu(lstm_input)

        # --- 3. LSTM ---
        gates_ih = torch.einsum('bnh,nhg->bng', lstm_input, self.w_lstm_ih) + self.b_lstm_ih
        gates_hh = torch.einsum('bnh,nhg->bng', self.h_state, self.w_lstm_hh) + self.b_lstm_hh
        gates = gates_ih + gates_hh

        ingate, forgetgate, cellgate, outgate = gates.chunk(4, dim=2)

        self.c_state = (torch.sigmoid(forgetgate) * self.c_state) + (torch.sigmoid(ingate) * torch.tanh(cellgate))
        self.h_state = torch.sigmoid(outgate) * torch.tanh(self.c_state)

        # --- 4. SENDER ---
        self.prev_outputs = torch.einsum('bnh,nhs->bns', self.h_state, self.w_sender) + self.b_sender

        # --- 5. OUTPUT ---
        idx_start = self.num_input + self.num_trans_static
        idx_end = idx_start + self.num_action
        action_out = self.prev_outputs[:, idx_start:idx_end, :]

        final = self.action_projection(action_out.reshape(batch_size, -1))
        return final


In [16]:
def plot_attention_matrix(model, filename="attention_matrix.png", title="Matrice d'Attention des Neurones"):
    """
    Version standard (Liste de neurones).
    """
    if hasattr(model, 'last_Q'):
        return plot_vectorized_attention_matrix(model, filename, title)

    model.eval()
    device = next(model.parameters()).device

    with torch.no_grad():
        all_queries = []
        all_keys = []
        for neuron in model.neurons:
            q, k = neuron.get_attention_params(batch_size=1)
            all_queries.append(q.unsqueeze(1))
            all_keys.append(k.unsqueeze(1))
        Q_matrix = torch.cat(all_queries, dim=1).to(device)
        K_matrix = torch.cat(all_keys, dim=1).to(device)
        scores = torch.bmm(Q_matrix, K_matrix.transpose(1, 2)) / (model.key_query_dim ** 0.5)
        attention_weights = torch.nn.functional.softmax(scores, dim=2)
        attention_matrix = attention_weights[0].cpu().numpy()

    _save_plot(attention_matrix, filename, title, model)

def plot_vectorized_attention_matrix(model, filename="attention_matrix.png", title="Matrice d'Attention"):
    """
    Version pour VectorizedIntelligentNetwork.
    """
    model.eval()

    if model.last_Q is None:
         print("Attention: Pas de Q/K en cache. Visualisation ignorée.")
         return

    with torch.no_grad():
        Q = model.last_Q[0:1]
        K = model.last_K[0:1]

        scores = torch.bmm(Q, K.transpose(1, 2)) / (model.key_query_dim ** 0.5)
        attention_weights = torch.nn.functional.softmax(scores, dim=2)
        attention_matrix = attention_weights[0].cpu().numpy()

    _save_plot(attention_matrix, filename, title, model)

def _save_plot(matrix, filename, title, model):
    plt.figure(figsize=(14, 12))

    labels = []
    # Detection des nombres de neurones selon le type de modele
    if hasattr(model, 'num_input'): # Vectorized
        n_in = model.num_input
        n_ts = getattr(model, 'num_trans_static', 0)
        n_td = getattr(model, 'num_trans_dynamic', 0)
        n_act = model.num_action

        # Si ancienne version vectorisée sans distinction static/dynamic
        if n_ts == 0 and n_td == 0:
             # Fallback sur l'attribut générique num_trans s'il existe
             n_ts = getattr(model, 'num_trans', 0)

        for i in range(n_in): labels.append(f"I{i}")
        for i in range(n_ts): labels.append(f"TS{i}")
        for i in range(n_act): labels.append(f"A{i}")
        for i in range(n_td): labels.append(f"TD{i}")
    else: # Standard
        n_in, n_tr, n_act = model.num_input_neurons, model.num_transmission_neurons, model.num_action_neurons
        for i in range(n_in): labels.append(f"I{i}")
        for i in range(n_tr): labels.append(f"T{i}")
        for i in range(n_act): labels.append(f"A{i}")

    sns.heatmap(matrix, xticklabels=labels, yticklabels=labels, cmap="viridis", vmin=0, vmax=1)

    plt.title(title)
    plt.xlabel("Émetteur (Source)")
    plt.ylabel("Receveur (Destination)")
    plt.tight_layout()

    if os.path.dirname(filename) and not os.path.exists(os.path.dirname(filename)):
        os.makedirs(os.path.dirname(filename))

    plt.savefig(filename)
    plt.close()
    print(f"Matrice d'attention sauvegardée dans '{filename}'")


In [17]:
# --- Config ---
BATCH_SIZE = 64
SEQ_LEN = 64
LEARNING_RATE = 0.001
EPOCHS = 30
HIDDEN_DIM = 256
KEY_QUERY_DIM = 256
SIGNAL_DIM = 256

NUM_INPUT = 5
NUM_TRANS_STATIC = 40  # Neurones à Key/Query fixes (Piliers)
NUM_TRANS_DYNAMIC = 20 # Neurones à Key/Query dynamiques (Liquides)
NUM_ACTION = 5

PRINT_EVERY = 100

def load_data(filepath):
    #with open(filepath, 'r', encoding='utf-8') as f:
    #    text = f.read()
    chars = sorted(list(set(text)))
    vocab_size = len(chars)
    char2idx = {ch: i for i, ch in enumerate(chars)}
    idx2char = {i: ch for i, ch in enumerate(chars)}
    return text, chars, char2idx, idx2char, vocab_size

def get_batch(text, char2idx, batch_size, seq_len, vocab_size):
    input_seqs = []
    target_seqs = []

    for _ in range(batch_size):
        start_idx = random.randint(0, len(text) - seq_len - 1)
        chunk = text[start_idx : start_idx + seq_len + 1]

        input_indices = [char2idx[c] for c in chunk[:-1]]
        target_indices = [char2idx[c] for c in chunk[1:]]

        input_tensor = torch.zeros(seq_len, vocab_size)
        for t, idx in enumerate(input_indices):
            input_tensor[t][idx] = 1.0

        input_seqs.append(input_tensor)
        target_seqs.append(torch.tensor(target_indices, dtype=torch.long))

    inputs = torch.stack(input_seqs)
    targets = torch.stack(target_seqs)
    return inputs, targets

def generate_text(model, start_str, char2idx, idx2char, vocab_size, length=100, temperature=0.8):
    model.eval()
    device = next(model.parameters()).device

    input_seq = torch.zeros(1, 1, vocab_size).to(device)
    model.reset_memory(batch_size=1)

    current_char = start_str[0]
    input_seq[0, 0, char2idx[current_char]] = 1.0

    generated_str = start_str

    with torch.no_grad():
        seed_tensor = torch.zeros(1, len(start_str), vocab_size).to(device)
        for t, char in enumerate(start_str):
            seed_tensor[0, t, char2idx[char]] = 1.0

        outputs = model(seed_tensor)
        last_output = outputs[:, -1, :]

        probs = torch.softmax(last_output / temperature, dim=1)
        next_idx = torch.multinomial(probs, 1).item()

        current_input = torch.zeros(1, 1, vocab_size).to(device)
        current_input[0, 0, next_idx] = 1.0

        for _ in range(length - 1):
            output = model(current_input.squeeze(1))

            probs = torch.softmax(output / temperature, dim=1)
            next_idx = torch.multinomial(probs, 1).item()
            next_char = idx2char[next_idx]
            generated_str += next_char

            current_input.zero_()
            current_input[0, 0, next_idx] = 1.0

    model.train()
    return generated_str

def train():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Utilisation du device : {device}")

    text, chars, char2idx, idx2char, vocab_size = load_data('test/input.txt')

    model = VectorizedIntelligentNetwork(
        input_size=vocab_size,
        output_size=vocab_size,
        num_input_neurons=NUM_INPUT,
        num_transmission_neurons=NUM_TRANS_STATIC,
        num_dynamic_neurons=NUM_TRANS_DYNAMIC,
        num_action_neurons=NUM_ACTION,
        neuron_hidden_dim=HIDDEN_DIM,
        key_query_dim=KEY_QUERY_DIM,
        signal_dim=SIGNAL_DIM
    ).to(device)

    print(f"Modèle Hybride créé avec {NUM_INPUT} In, {NUM_TRANS_STATIC} Stat, {NUM_TRANS_DYNAMIC} Dyn, {NUM_ACTION} Act.")

    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    criterion = nn.CrossEntropyLoss()

    print("Début de l'entraînement HYBRIDE VECTORISÉ...")
    if not os.path.exists("plots_vec"): os.makedirs("plots_vec")

    batches_per_epoch = 200

    for epoch in range(EPOCHS):
        start_time = time.time()
        total_loss = 0

        for i in range(batches_per_epoch):
            inputs, targets = get_batch(text, char2idx, BATCH_SIZE, SEQ_LEN, vocab_size)
            inputs, targets = inputs.to(device), targets.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs.reshape(-1, vocab_size), targets.reshape(-1))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()

            total_loss += loss.item()

            if i % PRINT_EVERY == 0:
                print(f"Batch {i}, Loss: {loss.item():.4f}")

        avg_loss = total_loss / batches_per_epoch
        duration = time.time() - start_time
        print(f"=== Epoch {epoch+1} terminée en {duration:.2f}s | Moyenne Loss: {avg_loss:.4f} ===")

        if (epoch + 1) % 5 == 0 or epoch == 0:
            plot_attention_matrix(model, filename=f"plots_vec/attention_epoch_{epoch+1}.png",
                                  title=f"Hybrid Attention - Epoch {epoch+1}")

        print("Génération :")
        print(generate_text(model, "The ", char2idx, idx2char, vocab_size))
        print("="*30)

if __name__ == "__main__":
    train()


Utilisation du device : cuda
Modèle Hybride créé avec 5 In, 40 Stat, 20 Dyn, 5 Act.
Début de l'entraînement HYBRIDE VECTORISÉ...
Batch 0, Loss: 4.1896
Batch 100, Loss: 3.3018
=== Epoch 1 terminée en 59.41s | Moyenne Loss: 3.2359 ===
Matrice d'attention sauvegardée dans 'plots_vec/attention_epoch_1.png'
Génération :
The  rers ee aesd.
WTIwNdTEIAHEI

T
 :ehtee ce  enao. 
,wsud shweri  suvui  url teyl ah ere wowct
  uer
Batch 0, Loss: 3.0004
Batch 100, Loss: 2.9316
=== Epoch 2 terminée en 59.49s | Moyenne Loss: 2.8843 ===
Génération :
The aseorh .oysnu
li vook
,tsovb eerslsht   oe ruesi corrnt
hool  lrenkn foam sibees paupeus, antei dht
Batch 0, Loss: 2.8010
Batch 100, Loss: 2.7295
=== Epoch 3 terminée en 59.25s | Moyenne Loss: 2.7371 ===
Génération :
The oaee
 rfrrltncr lostens  oir sesyen,iAhole,hst aht wieh artnAd tsedc.r
ttcbae'i, Fntyudle
ufnrmrtnn
Batch 0, Loss: 2.6644
Batch 100, Loss: 2.6235
=== Epoch 4 terminée en 59.47s | Moyenne Loss: 2.6143 ===
Génération :
The uredorn,dnrtaIgK

# MultiTasking

Notre modèle performe très bien sur Shakespear pour un RNN, et il semble converger vers un résultat plus que satisfaisant. Poour l'instant, il est au mieux aussi bien qu'un transformer.

Voyons maintenant si on peut lui apprendre à faire une addition..

Sans qu'il oublie à écrire du Shakespear !

In [ ]:
f.readlines()

In [24]:
def generate_equation():
    """Génère une équation simple 'a+b=c'."""
    # Opérations : +, -, * (pour varier un peu)
    op = random.choice(['+', '-'])

    if op == '+':
        a = random.randint(0, 999)
        b = random.randint(0, 999)
        c = a + b
        return f"MATH: {a}+{b}={c}" # On ajoute un préfixe clair pour aider au début
    else:
        a = random.randint(0, 999)
        b = random.randint(0, a) # Pour rester positif simple
        c = a - b
        return f"MATH: {a}-{b}={c}"

def create_mixed_dataset(output_path):
    mixed_lines = []

    with open('input.txt', 'r', encoding='utf-8') as f:
      shakespeare_lines = f.readlines()
    # On garde les lignes non vides
    shakespeare_lines = [line.strip() for line in shakespeare_lines if line.strip()]

    # On limite la taille si c'est trop gros (pour équilibrer)
    # Disons qu'on veut 50% maths / 50% texte

    for line in shakespeare_lines:
        mixed_lines.append(line)
        mixed_lines.append(generate_equation())

    # Écriture
    with open(output_path, 'w', encoding='utf-8') as f:
        for line in mixed_lines:
            f.write(line + '\n')

    print(f"Dataset mixte généré : {len(mixed_lines)} lignes dans {output_path}")
    print("Exemple de contenu :")
    for l in mixed_lines[:6]:
        print(f"  {l}")

if __name__ == "__main__":
    create_mixed_dataset('input2.txt')



Dataset mixte généré : 65554 lignes dans input2.txt
Exemple de contenu :
  First Citizen:
  MATH: 103-26=77
  Before we proceed any further, hear me speak.
  MATH: 55+426=481
  All:
  MATH: 3+968=971


In [25]:
# --- Config ---
BATCH_SIZE = 64
SEQ_LEN = 64
LEARNING_RATE = 0.001
EPOCHS = 30
HIDDEN_DIM = 256
KEY_QUERY_DIM = 256
SIGNAL_DIM = 256

NUM_INPUT = 5
NUM_TRANS_STATIC = 40  # Neurones à Key/Query fixes (Piliers)
NUM_TRANS_DYNAMIC = 20 # Neurones à Key/Query dynamiques (Liquides)
NUM_ACTION = 5

PRINT_EVERY = 100

def load_data(filepath):
    with open('input2.txt', 'r', encoding='utf-8') as f:
        text = f.read()
    chars = sorted(list(set(text)))
    vocab_size = len(chars)
    char2idx = {ch: i for i, ch in enumerate(chars)}
    idx2char = {i: ch for i, ch in enumerate(chars)}
    return text, chars, char2idx, idx2char, vocab_size

def get_batch(text, char2idx, batch_size, seq_len, vocab_size):
    input_seqs = []
    target_seqs = []

    for _ in range(batch_size):
        start_idx = random.randint(0, len(text) - seq_len - 1)
        chunk = text[start_idx : start_idx + seq_len + 1]

        input_indices = [char2idx[c] for c in chunk[:-1]]
        target_indices = [char2idx[c] for c in chunk[1:]]

        input_tensor = torch.zeros(seq_len, vocab_size)
        for t, idx in enumerate(input_indices):
            input_tensor[t][idx] = 1.0

        input_seqs.append(input_tensor)
        target_seqs.append(torch.tensor(target_indices, dtype=torch.long))

    inputs = torch.stack(input_seqs)
    targets = torch.stack(target_seqs)
    return inputs, targets

def generate_text(model, start_str, char2idx, idx2char, vocab_size, length=100, temperature=0.8):
    model.eval()
    device = next(model.parameters()).device

    input_seq = torch.zeros(1, 1, vocab_size).to(device)
    model.reset_memory(batch_size=1)

    current_char = start_str[0]
    input_seq[0, 0, char2idx[current_char]] = 1.0

    generated_str = start_str

    with torch.no_grad():
        seed_tensor = torch.zeros(1, len(start_str), vocab_size).to(device)
        for t, char in enumerate(start_str):
            seed_tensor[0, t, char2idx[char]] = 1.0

        outputs = model(seed_tensor)
        last_output = outputs[:, -1, :]

        probs = torch.softmax(last_output / temperature, dim=1)
        next_idx = torch.multinomial(probs, 1).item()

        current_input = torch.zeros(1, 1, vocab_size).to(device)
        current_input[0, 0, next_idx] = 1.0

        for _ in range(length - 1):
            output = model(current_input.squeeze(1))

            probs = torch.softmax(output / temperature, dim=1)
            next_idx = torch.multinomial(probs, 1).item()
            next_char = idx2char[next_idx]
            generated_str += next_char

            current_input.zero_()
            current_input[0, 0, next_idx] = 1.0

    model.train()
    return generated_str

def train():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Utilisation du device : {device}")

    text, chars, char2idx, idx2char, vocab_size = load_data('test/input.txt')

    model = VectorizedIntelligentNetwork(
        input_size=vocab_size,
        output_size=vocab_size,
        num_input_neurons=NUM_INPUT,
        num_transmission_neurons=NUM_TRANS_STATIC,
        num_dynamic_neurons=NUM_TRANS_DYNAMIC,
        num_action_neurons=NUM_ACTION,
        neuron_hidden_dim=HIDDEN_DIM,
        key_query_dim=KEY_QUERY_DIM,
        signal_dim=SIGNAL_DIM
    ).to(device)

    print(f"Modèle Hybride créé avec {NUM_INPUT} In, {NUM_TRANS_STATIC} Stat, {NUM_TRANS_DYNAMIC} Dyn, {NUM_ACTION} Act.")

    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    criterion = nn.CrossEntropyLoss()

    print("Début de l'entraînement HYBRIDE VECTORISÉ...")
    if not os.path.exists("plots_vec"): os.makedirs("plots_vec")

    batches_per_epoch = 200

    for epoch in range(EPOCHS):
        start_time = time.time()
        total_loss = 0

        for i in range(batches_per_epoch):
            inputs, targets = get_batch(text, char2idx, BATCH_SIZE, SEQ_LEN, vocab_size)
            inputs, targets = inputs.to(device), targets.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs.reshape(-1, vocab_size), targets.reshape(-1))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()

            total_loss += loss.item()

            if i % PRINT_EVERY == 0:
                print(f"Batch {i}, Loss: {loss.item():.4f}")

        avg_loss = total_loss / batches_per_epoch
        duration = time.time() - start_time
        print(f"=== Epoch {epoch+1} terminée en {duration:.2f}s | Moyenne Loss: {avg_loss:.4f} ===")

        if (epoch + 1) % 5 == 0 or epoch == 0:
            plot_attention_matrix(model, filename=f"plots_vec/attention_epoch_{epoch+1}.png",
                                  title=f"Hybrid Attention - Epoch {epoch+1}")

        print("Génération :")
        print(generate_text(model, "The ", char2idx, idx2char, vocab_size))
        print("="*30)

if __name__ == "__main__":
    train()


Utilisation du device : cuda
Modèle Hybride créé avec 5 In, 40 Stat, 20 Dyn, 5 Act.
Début de l'entraînement HYBRIDE VECTORISÉ...
Batch 0, Loss: 4.3490
Batch 100, Loss: 3.2581
=== Epoch 1 terminée en 59.84s | Moyenne Loss: 3.2624 ===
Matrice d'attention sauvegardée dans 'plots_vec/attention_epoch_1.png'
Génération :
The vi r   e ng.tMeMATH: 081+227=1240
TUi leeeeegi rosa e  vevthoeeessoeee

MATH: 397+87==57

alh  ts t
Batch 0, Loss: 2.5821
Batch 100, Loss: 2.7476
=== Epoch 2 terminée en 59.97s | Moyenne Loss: 2.6908 ===
Génération :
The oint
sA Iew tehn
aAhner  eue terlt eahti
eM,TA:AAMATH: 377+671=782
Aod'  ohthtt soi  eaosea  ieedtH
Batch 0, Loss: 2.5299
Batch 100, Loss: 2.4579
=== Epoch 3 terminée en 59.78s | Moyenne Loss: 2.4478 ===
Génération :
The is  ao te kont.fMhTs rameeg
gA Mo7l-nM Tnodn  ear iom  oste mo .algttsMMMH8:drO'Esw:aMhT::M+T2:YA Y
Batch 0, Loss: 2.4465
Batch 100, Loss: 2.3564
=== Epoch 4 terminée en 59.69s | Moyenne Loss: 2.3808 ===
Génération :
The lo  eot ia err s